In [11]:
# Cell 1 — Import Required Libraries and Configure LLM Access
import ollama

response = ollama.chat(
    model="llama3.1",
    messages=[
        {
            "role": "user",
            "content": "Why is the sky blue?"
        }
    ]
)

print(response["message"]["content"])

The sky appears blue to us because of a phenomenon called Rayleigh scattering, named after the British physicist Lord Rayleigh, who first described it in the late 19th century. Here's a simplified explanation:

1. **Sunlight and Its Composition**: Sunlight is made up of a broad spectrum of electromagnetic radiation, including various wavelengths of visible light. When sunlight enters Earth's atmosphere, it encounters tiny molecules of gases such as nitrogen (N2) and oxygen (O2).

2. **Scattering of Light**: These gas molecules scatter the light in all directions. However, they scatter shorter (blue) wavelengths more than longer (red) wavelengths. This is the principle behind Rayleigh scattering.

3. **Why Blue and Not Red?**: The reason why the sky appears blue is because the blue light is scattered more than the other colors by the Earth's atmosphere. As a result, when we look up at the sky, the blue light is what we see the most because it has been scattered in all directions by the 

In [12]:
# Cell 2 — Load and Validate Cleaned Data
import pandas as pd
df = pd.read_csv("../data/processed/customer_support_tickets_clean.csv")

print(f"Duplicated tickets: {df.duplicated().sum()}")

invalid_type = (
    df["Ticket Type"]
    .fillna("")
    .str.strip()
    .eq("")
    .sum()
)

print(f"Invalid or empty Ticket Type values: {invalid_type}")

invalid_description = (
    df["Ticket Description"]
    .fillna("")
    .str.strip()
    .eq("")
    .sum()
)

print(f"Invalid or empty Ticket Description values: {invalid_description}")

Duplicated tickets: 0
Invalid or empty Ticket Type values: 0
Invalid or empty Ticket Description values: 0


In [13]:
# Cell 3 — Define Allowed Ticket Categories
ticket_categories = [
    "Technical Issue",
    "Refund Request",
    "Cancellation Request",
    "Billing Inquiry",
    "Product Inquiry"
]

In [14]:
# Cell 4 — Create the Structured Ticket Analysis Prompt

def create_prompt(ticket_description):
    prompt = f"""
Analyze the following customer-support ticket.

Ticket description:
{ticket_description}

Classify the ticket using exactly one of these categories:
- Technical Issue
- Refund Request
- Cancellation Request
- Billing Inquiry
- Product Inquiry

Return the analysis as JSON using exactly these four fields:

{{
    "Summary": "A short description of the customer's problem",
    "Category": "One of the five allowed ticket categories",
    "Sentiment": "The customer's expressed sentiment",
    "Urgency": "The estimated urgency of the problem"
}}

Return only the JSON object and no additional text.
"""
    return prompt

In [15]:
# Cell 5 — Create analyze_ticket() Function

import json

def analyze_ticket(ticket_description):
    if ticket_description.strip() == "":
        return None

    prompt = create_prompt(ticket_description)

    response = ollama.chat(
        model="llama3.1",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    response_text = response["message"]["content"]

    analysis = json.loads(response_text)

    return analysis

In [16]:
# Cell 6 — Test Structured Analysis with One Ticket
ticket_test = df["Ticket Description"].iloc[0]
llm_resp = analyze_ticket(ticket_test)
print(llm_resp)
print(type(llm_resp))

{'Summary': 'Customer is experiencing an issue with the product they purchased', 'Category': 'Technical Issue', 'Sentiment': 'Neutral', 'Urgency': 'Medium'}
<class 'dict'>


In [17]:
# Cell 7 — Access Individual Analysis Fields

print("Summary:", llm_resp["Summary"])
print("Category:", llm_resp["Category"])
print("Sentiment:", llm_resp["Sentiment"])
print("Urgency:", llm_resp["Urgency"])

Summary: Customer is experiencing an issue with the product they purchased
Category: Technical Issue
Sentiment: Neutral
Urgency: Medium


In [18]:
# Cell 8 — Test Structured Analysis with Multiple Tickets

multiple_tickets = df["Ticket Description"].sample(
    10,
    random_state=42
)

analysis_results = []

for ticket in multiple_tickets:
    result = analyze_ticket(ticket)

    analysis_results.append({
        "Ticket Description": ticket,
        "Summary": result["Summary"],
        "Category": result["Category"],
        "Sentiment": result["Sentiment"],
        "Urgency": result["Urgency"]
    })

    print(result)

{'Summary': 'Issue with {product_purchased} app', 'Category': 'Technical Issue', 'Sentiment': 'Neutral', 'Urgency': 'Medium'}
{'Summary': 'Customer is experiencing issues connecting {product_purchased} to home Wi-Fi network', 'Category': 'Technical Issue', 'Sentiment': 'Neutral', 'Urgency': 'Medium'}
{'Summary': 'Concerns about security of {product_purchased} and data safety', 'Category': 'Technical Issue', 'Sentiment': 'Neutral/Concerned', 'Urgency': 'Medium'}
{'Summary': 'Error message is displaying on {product_purchased} screen', 'Category': 'Technical Issue', 'Sentiment': 'Neutral', 'Urgency': 'Medium'}
{'Summary': "Customer's product has decreased battery life and they want a price increase to facilitate a return.", 'Category': 'Refund Request', 'Sentiment': 'Negative', 'Urgency': 'Medium'}
{'Summary': 'Data loss issue with product, files and documents disappeared', 'Category': 'Technical Issue', 'Sentiment': 'Frustrated/Concerned', 'Urgency': 'High'}
{'Summary': 'Peculiar error m

In [19]:
# Cell 9 — Convert Multiple LLM Results into a DataFrame

responses = pd.DataFrame(analysis_results)

display(responses)

,Ticket Description,Summary,Category,Sentiment,Urgency
0,I'm having an issue with the {product_purchase...,Issue with {product_purchased} app,Technical Issue,Neutral,Medium
1,I'm having trouble connecting my {product_purc...,Customer is experiencing issues connecting {pr...,Technical Issue,Neutral,Medium
2,I'm having an issue with the {product_purchase...,Concerns about security of {product_purchased}...,Technical Issue,Neutral/Concerned,Medium
3,I'm having an issue with the {product_purchase...,Error message is displaying on {product_purcha...,Technical Issue,Neutral,Medium
4,I'm having an issue with the {product_purchase...,Customer's product has decreased battery life ...,Refund Request,Negative,Medium
5,I've encountered a data loss issue with my {pr...,"Data loss issue with product, files and docume...",Technical Issue,Frustrated/Concerned,High
6,I'm having an issue with the {product_purchase...,Peculiar error message on {product_purchased} ...,Technical Issue,Neutral,Medium
7,I'm having an issue with the {product_purchase...,Customer is experiencing a decrease in battery...,Technical Issue,Neutral,Medium
8,I'm encountering a software bug in the {produc...,"Software application crashes, seeking updates ...",Technical Issue,Neutral,High
9,I'm having an issue with the {product_purchase...,The customer is experiencing an issue with the...,Technical Issue,Neutral,Medium


In [ ]:
# Cell 10 — Validate Structured LLM Outputs

invalid_results = 0

for result in analysis_results:
    valid_category = result["Category"] in ticket_categories
    valid_summary = bool(result["Summary"])
    valid_sentiment = bool(result["Sentiment"])
    valid_urgency = bool(result["Urgency"])

    if not (
        valid_category
        and valid_summary
        and valid_sentiment
        and valid_urgency
    ):
        invalid_results += 1

print("Number of invalid structured results:", invalid_results)

Number of invalid structured results: 0


: 

## Day 29 Conclusions

### Structured LLM Ticket Analysis

Day 29 converted the free-text LLM analysis from Day 28 into structured data that can be accessed and processed directly with Python.

The `analyze_ticket()` function now performs the following pipeline:

**Ticket Description → Structured Prompt → Llama 3.1 → JSON Response → Python Dictionary**

The function asks the LLM to return exactly four fields:

- **Summary** — a short description of the customer's problem
- **Category** — one of the five predefined Ticket Type categories
- **Sentiment** — the customer's expressed sentiment
- **Urgency** — the estimated urgency of the problem

### JSON Conversion

The LLM response is initially returned as text. `json.loads()` converts the JSON-formatted response into a Python dictionary.

The single-ticket test successfully returned a Python dictionary containing all four required fields. This made it possible to access individual results using dictionary keys such as:

- `llm_resp["Summary"]`
- `llm_resp["Category"]`
- `llm_resp["Sentiment"]`
- `llm_resp["Urgency"]`

For the first tested ticket, the LLM returned:

- **Category:** Technical Issue
- **Sentiment:** Neutral
- **Urgency:** High

### Multiple-Ticket Analysis

The structured analysis was tested on 10 sampled ticket descriptions.

Each LLM response was stored together with its original ticket description and converted into a Pandas DataFrame containing:

**Ticket Description | Summary | Category | Sentiment | Urgency**

The model returned several different categories, sentiments, and urgency levels across the sample. Most of the sampled tickets were categorized as **Technical Issue**, while other examples were categorized as **Product Inquiry** and **Refund Request**.

The sentiment outputs included values such as:

- Neutral
- Frustrated
- Negative
- Concerned
- Neutral/Frustrated
- Concerned/Frustrated

Urgency values in the sample were **Medium** or **High**.

### Structured Output Validation

All 10 sampled responses were checked to verify that:

1. The Category belonged to one of the five allowed Ticket Type categories.
2. Summary was not empty.
3. Sentiment was not empty.
4. Urgency was not empty.

The validation found:

**Invalid structured results: 0**

Therefore, all 10 tested responses satisfied the required output structure.

However, this validation does **not** determine whether the LLM classifications were correct. A response can contain a valid allowed category while still assigning the wrong category to a ticket. Classification accuracy must be evaluated separately against the actual Ticket Type labels.

### Observations

The LLM successfully produced machine-readable results for all tickets in the tested sample. The structured format makes the results easier to store in a DataFrame and prepares them for later evaluation.

The sentiment labels were not completely standardized. For example, the LLM produced both individual labels such as `Frustrated` and combined labels such as `Neutral/Frustrated`. If sentiment is used as a categorical variable later, a fixed set of allowed sentiment labels may be needed.

Some source ticket descriptions also contain placeholders such as `{product_purchased}` and `{error_message}`. These template-like descriptions may affect both traditional ML and LLM classification and should be considered when interpreting later results.

### Conclusion

Day 29 successfully transformed the LLM ticket analyzer from a free-text prototype into a reusable structured analysis pipeline.

The resulting workflow is:

**New Ticket → `analyze_ticket()` → Llama 3.1 → JSON → Python Dictionary → DataFrame**

The structured output can now be processed programmatically rather than manually reading the LLM response.

The next stages can use this structured pipeline to generate results on a controlled set of tickets and eventually compare the LLM's Ticket Type classifications with the traditional TF-IDF and Logistic Regression baseline.